# 01 · Prepare RDD2022 (India subset)

> **OWNER:** Member A (M1 · road defects)
> **PREREQUISITES:** `00_setup_verify.ipynb` all-green.
> **EXPECTED RUNTIME:** 1-3 hours, almost entirely the download+extract — the
> conversion/split/contact-sheet steps run in a few minutes once the raw data
> is on disk.
> **OUTPUTS:** `data/rdd2022_india/{images,labels}/{train,val,test}/` + `data.yaml`,
> a contact sheet PNG, and printed class-balance / on-disk-size reports.

Source: figshare RDD2022, DOI [10.6084/m9.figshare.21431547](https://doi.org/10.6084/m9.figshare.21431547),
and [github.com/sekilab/RoadDamageDetector](https://github.com/sekilab/RoadDamageDetector)
(loader scripts + the same data).

**Next notebook:** `02_train_road_damage.ipynb` (yours too).

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 1 — Get the archive on disk (automated — figshare is the only working source)

**The per-country S3 bucket is dead.** `RDD2022_India.zip` (502.3 MB) is listed
in the sekilab README at
`bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/`,
but every file in that bucket returned `403 Access Denied` when checked on
2026-09-05 — tested both `RDD2022_India.zip` and `RDD2022_Japan.zip` to confirm
it's the whole bucket, not a renamed path. If the per-country links come back
later, prefer them (much smaller, no selective-extract step needed) — but as
of this writing there is no smaller path.

**Figshare (DOI [10.6084/m9.figshare.21431547](https://doi.org/10.6084/m9.figshare.21431547))
is the only source that responds**, and it hosts exactly one file — the full
six-country archive:

| | |
|---|---|
| file | `RDD2022_released_through_CRDDC2022.zip` |
| size | 13,264,172,619 bytes (12.35 GB) |
| md5 | `b62bd51d2ffcfaa76c60f234f0cc2bb3` |
| download URL | https://ndownloader.figshare.com/files/38030910 |

The cell below downloads it straight to **local/ephemeral disk** — never to
Drive — extracts only the `India/` members, then deletes the 12.35 GB archive
immediately. Only the small, prepared output (`data/rdd2022_india/`, a few
hundred MB) is written to the Drive-backed `DATA_ROOT`, so a dropped Colab
session loses at most a re-download + re-extract (~15-20 min), never the
converted dataset.

In [ ]:
import hashlib
import shutil
import urllib.request

FIGSHARE_URL = "https://ndownloader.figshare.com/files/38030910"
FIGSHARE_SIZE_BYTES = 13_264_172_619
FIGSHARE_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

# Local/ephemeral disk regardless of environment — on Colab this is the VM's
# own disk (/content), never the Drive mount, so a 12.35 GB download+extract
# never touches Drive quota or Drive's (much slower) FUSE I/O.
RAW_DIR = Path("/content/rdd2022_scratch") if env["colab"] else (DATA_ROOT / "raw" / "rdd2022")
RAW_DIR.mkdir(parents=True, exist_ok=True)
archive_path = RAW_DIR / "RDD2022_released_through_CRDDC2022.zip"


def _free_gb(path):
    return shutil.disk_usage(path).free / (1024**3)


print(f"free disk before download: {_free_gb(RAW_DIR):.1f} GB")

if archive_path.exists() and archive_path.stat().st_size == FIGSHARE_SIZE_BYTES:
    print(f"{archive_path} already present at the expected size — skipping download")
else:
    print(f"downloading {FIGSHARE_URL}")
    print(f"  -> {archive_path}  ({FIGSHARE_SIZE_BYTES / 1024**3:.2f} GB, ~15-25 min on Colab's network)")

    def _report(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if block_num % 4000 == 0 or downloaded >= total_size:
            pct = min(100.0, downloaded / total_size * 100) if total_size > 0 else 0.0
            print(f"  {downloaded / 1024**3:.2f} / {total_size / 1024**3:.2f} GB ({pct:.1f}%)")

    urllib.request.urlretrieve(FIGSHARE_URL, archive_path, reporthook=_report)

actual_size = archive_path.stat().st_size
if actual_size != FIGSHARE_SIZE_BYTES:
    raise RuntimeError(
        f"downloaded size {actual_size} != expected {FIGSHARE_SIZE_BYTES} — "
        "partial/corrupt download, delete the file and re-run this cell rather than trusting it"
    )

print("verifying md5 (~1 min for a 12 GB file)...")
_h = hashlib.md5()
with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        _h.update(chunk)
actual_md5 = _h.hexdigest()
if actual_md5 != FIGSHARE_MD5:
    raise RuntimeError(f"md5 mismatch: got {actual_md5}, expected {FIGSHARE_MD5} — archive is corrupt, delete and re-download")
print(f"md5 verified: {actual_md5}")

print(f"free disk after download: {_free_gb(RAW_DIR):.1f} GB")

## Step 1b — selective extract: India/ only, then delete the archive

The archive has six countries in it; we only want `India/`. `zipfile` lets us
extract individual members without ever unpacking Japan/Norway/Czechia/USA/China
to disk. The archive is deleted immediately afterward — at no point do we hold
both "the full 12.35 GB zip" and "all six countries extracted" on disk at once.

In [ ]:
import zipfile

EXTRACT_ROOT = RAW_DIR / "extracted"
EXTRACT_ROOT.mkdir(exist_ok=True)

print(f"free disk before extraction: {_free_gb(RAW_DIR):.1f} GB")

with zipfile.ZipFile(archive_path) as zf:
    all_members = zf.namelist()
    india_members = [m for m in all_members if "india" in m.lower()]
    print(f"archive has {len(all_members)} total members; extracting {len(india_members)} India-only members")
    if not india_members:
        raise RuntimeError(
            "No member path contains 'India' — inspect zf.namelist() manually, "
            "the archive's internal naming may have changed since this was written."
        )
    for i, member in enumerate(india_members):
        zf.extract(member, EXTRACT_ROOT)
        if (i + 1) % 5000 == 0 or (i + 1) == len(india_members):
            print(f"  extracted {i + 1}/{len(india_members)}")

print(f"free disk after extraction: {_free_gb(RAW_DIR):.1f} GB")

In [ ]:
archive_size_gb = archive_path.stat().st_size / 1024**3
print(f"deleting {archive_path} ({archive_size_gb:.2f} GB) — India/ is extracted, "
      "the other five countries were never written to disk")
archive_path.unlink()

print(f"free disk after deleting archive: {_free_gb(RAW_DIR):.1f} GB")

## Step 1c — locate the images/ and annotations/ folders

RDD2022's directory structure is `RDD2022/India/{train,test}/...` — `train/`
has both `images/` and `annotations/xmls/`; `test/` has images only (unlabeled,
CRDDC-competition holdout). We need `train/`'s pair. A naive "first directory
whose name contains 'image'" search can resolve to `test/images` instead
(alphabetically `test` sorts before `train`), which would silently pair every
image with zero XML annotations — so the search below explicitly prefers a
path containing `train`.

In [ ]:
def _find_dir(root, name_contains, prefer_contains=None):
    candidates = [p for p in root.rglob("*") if p.is_dir() and name_contains in p.name.lower()]
    if prefer_contains:
        preferred = [p for p in candidates if prefer_contains in str(p).lower()]
        if preferred:
            candidates = preferred
    return candidates[0] if candidates else None


IMAGES_DIR = _find_dir(EXTRACT_ROOT, "image", prefer_contains="train")
ANNOTATIONS_DIR = _find_dir(EXTRACT_ROOT, "xml", prefer_contains="train") or _find_dir(EXTRACT_ROOT, "annotation", prefer_contains="train")

print(f"IMAGES_DIR:      {IMAGES_DIR}")
print(f"ANNOTATIONS_DIR: {ANNOTATIONS_DIR}")

if IMAGES_DIR is None or ANNOTATIONS_DIR is None:
    print()
    print("Could not auto-locate both directories. List EXTRACT_ROOT and set them by hand:")
    print(f"  list(EXTRACT_ROOT.rglob('*'))[:40]  # EXTRACT_ROOT = {EXTRACT_ROOT}")

## Step 2 — Inspect before converting

Image/XML pairing, resolution distribution, and — **printed prominently** —
instances per class. RDD2022's D40 (POTHOLE) is the minority class, and
POTHOLE is the class the entire pitch is about. If the ratio below is severe,
notebook 02's second training run addresses it explicitly — but you need the
real number first.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

from PIL import Image

_image_stems = {p.stem for p in IMAGES_DIR.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
_xml_stems = {p.stem for p in ANNOTATIONS_DIR.glob("*.xml")}

orphan_images = sorted(_image_stems - _xml_stems)
orphan_xml = sorted(_xml_stems - _image_stems)
print(f"images: {len(_image_stems)}   xml: {len(_xml_stems)}")
print(f"orphan images (no xml): {len(orphan_images)}  e.g. {orphan_images[:5]}")
print(f"orphan xml (no image):  {len(orphan_xml)}  e.g. {orphan_xml[:5]}")

resolutions = Counter()
class_counts = Counter()
_paired = sorted(_image_stems & _xml_stems)
for stem in _paired:
    img_path = next(p for p in IMAGES_DIR.iterdir() if p.stem == stem)
    with Image.open(img_path) as img:
        resolutions[img.size] += 1
    root = ET.parse(ANNOTATIONS_DIR / f"{stem}.xml").getroot()
    for obj in root.findall("object"):
        name_el = obj.find("name")
        if name_el is not None and name_el.text:
            class_counts[name_el.text.strip()] += 1

print()
print("resolution distribution (top 5):")
for res, n in resolutions.most_common(5):
    print(f"  {res}: {n} images")

print()
print("=" * 60)
print("CLASS BALANCE (all D-codes present in the raw XML, before filtering to the frozen 4):")
total = sum(class_counts.values()) or 1
for name, n in class_counts.most_common():
    marker = "  <-- frozen class" if name in ("D00", "D10", "D20", "D40") else ""
    print(f"  {name:12s} {n:6d}  ({100 * n / total:5.1f}%){marker}")
print("=" * 60)
d40 = class_counts.get("D40", 0)
d00 = class_counts.get("D00", 0)
if d40 and d00:
    print(f"D40 (POTHOLE) is {d40 / max(d00, 1):.2f}x the count of D00 (LONGITUDINAL_CRACK) — "
          f"{'a severe imbalance' if d40 / max(d00, 1) < 0.3 else 'imbalanced, note it'}. "
          "Notebook 02 trains a second run addressing this directly.")

## Step 3 — Convert VOC -> YOLO, frozen indices

Only the four frozen classes (D00/D10/D20/D40) are kept — every other D-code
present in the raw data is reported as `unknown_classes` and dropped, not
silently folded into a class it doesn't belong to.

In [ ]:
from common import constants, voc_to_yolo

CLASS_MAP = {code_name: idx for idx, code_name in constants.RDD_CLASSES.items()}  # {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
print(f"class_map (VOC name -> frozen index): {CLASS_MAP}")

CONVERTED_LABELS_DIR = RAW_DIR / "labels_yolo"  # scratch — only the materialized split below persists to Drive
report = voc_to_yolo.convert_voc_dir(
    images_dir=IMAGES_DIR,
    annotations_dir=ANNOTATIONS_DIR,
    class_map=CLASS_MAP,
    output_labels_dir=CONVERTED_LABELS_DIR,
)
report.print_summary()

## Step 4 — Stratified 80/10/10 split (seed=42)

Stratified by each image's rarest present class so D40 (POTHOLE) doesn't concentrate in one split.

In [ ]:
from common import splits

image_class_map = {}
for stem in _paired:
    label_path = CONVERTED_LABELS_DIR / f"{stem}.txt"
    classes = set()
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if line.strip():
                classes.add(int(line.split()[0]))
    image_class_map[stem] = classes

data_splits = splits.stratified_split(image_class_map, ratios=(0.8, 0.1, 0.1), seed=42)
splits.report_split_balance(image_class_map, data_splits, constants.RDD_DETECTION_NAMES)

## Step 5 — Materialize the split + write data.yaml

In [ ]:
import yaml

OUTPUT_ROOT = DATA_ROOT / "rdd2022_india"
splits.materialize_split(
    image_class_map=image_class_map,
    splits=data_splits,
    images_src=IMAGES_DIR,
    labels_src=CONVERTED_LABELS_DIR,
    output_root=OUTPUT_ROOT,
)

data_yaml = {
    "path": str(OUTPUT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": constants.RDD_DETECTION_NAMES,
}
constants.assert_class_order(data_yaml["names"], constants.RDD_DETECTION_NAMES, "rdd")

data_yaml_path = OUTPUT_ROOT / "data.yaml"
data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(f"wrote {data_yaml_path}")
print(data_yaml_path.read_text())

print(f"free disk (local/scratch) after materializing to Drive: {_free_gb(RAW_DIR):.1f} GB")

## Step 6 — Contact sheet — LOOK AT THIS BEFORE ANYTHING ELSE

12 random training images with boxes drawn. **Offset or inverted boxes mean
the VOC->YOLO conversion is broken** — training will run happily on broken
boxes and produce a model with a normal-looking loss curve that is useless in
the field. Ten seconds here saves a day of debugging why the trained model is
garbage.

In [ ]:
from common import contact_sheet

sheet_path = contact_sheet.render_contact_sheet(
    images_dir=OUTPUT_ROOT / "images" / "train",
    labels_dir=OUTPUT_ROOT / "labels" / "train",
    class_names=constants.RDD_DETECTION_NAMES,
    output_path=OUTPUT_ROOT / "contact_sheet.png",
    n=12,
)

from IPython.display import Image as IPImage, display

display(IPImage(filename=str(sheet_path)))

## Step 7 — gitignore check + on-disk size

In [ ]:
def _dir_size_gb(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / (1024**3)


dataset_size_gb = _dir_size_gb(OUTPUT_ROOT)
raw_size_gb = _dir_size_gb(RAW_DIR)
print(f"prepared dataset ({OUTPUT_ROOT}): {dataset_size_gb:.2f} GB")
print(f"raw download+extract ({RAW_DIR}):  {raw_size_gb:.2f} GB")
print("(RAW_DIR is local/ephemeral scratch, not Drive — it disappears when the Colab runtime recycles; "
      "only the prepared dataset above is Drive-backed and durable)")

gitignore_path = REPO_ROOT / ".gitignore"
gitignore_text = gitignore_path.read_text()
if "data/rdd2022_india/" in gitignore_text:
    print(".gitignore already excludes data/rdd2022_india/ — nothing to do")
else:
    print("WARNING: data/rdd2022_india/ is not in .gitignore — add it before `git add`ing anything")

---
### What this notebook produced
- `data/rdd2022_india/{images,labels}/{train,val,test}/` + `data.yaml` — on Colab this
  is Drive-backed (`DATA_ROOT` under `MyDrive/urban-twin-ml/data/`), so it survives a
  dropped runtime; `02_train_road_damage.ipynb` reads it straight from there, no re-download
- A contact sheet PNG you looked at and confirmed looks correct
- The real D40/D00 class-balance ratio, printed prominently
- The raw 12.35 GB figshare archive and its extracted-then-converted intermediates are gone —
  they lived only in local/ephemeral scratch (`RAW_DIR`), deleted or left behind on the VM's
  disk, never on Drive

### Next
`02_train_road_damage.ipynb` (yours too — M1).